# Simulating a VLA observation

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/simulation/simulate_processing_set_vla.ipynb)

This tutorial simulates the visibilities of a point source observed with the VLA in its D
configuration and images the result with AstroVIPER.  It is the AstroVIPER port of the SIRIUS
`simple_simulation` and `evla_simulation` notebooks.

---

## Assumptions and Background

The simulator (`astroviper.distributed_applications.simulation.simulate_processing_set`) evaluates
the measurement equation for a list of **point sources** and a list of **antenna beam models**:

$$V_{a_1 a_2}(t, \nu) = \sum_s \mathbf{M}(\mathbf{J}_{a_1}, \mathbf{J}_{a_2}) \,
\mathbf{S}_s(t, \nu) \; \frac{e^{2\pi i\, \mathbf{k}_s(t) \cdot \mathbf{uvw}_{a_1 a_2}(t)\, \nu / c}}{n_s(t)}$$

- $\mathbf{S}_s$ is the 4-correlation flux of source $s$ (``RR, RL, LR, LL`` or ``XX, XY, YX, YY``).
- $\mathbf{J}_a$ is the antenna voltage beam (Jones vector) sampled at the source direction relative
  to the pointing of antenna $a$; $\mathbf{M}$ is the Mueller matrix (elements chosen with
  ``beam_params["mueller_selection"]``).
- ``uvw`` follow the MSv4 convention (baseline = antenna2 - antenna1).

The output is a **Measurement Set v4 processing set** (``*.ps.zarr``) written chunk-wise by
GraphVIPER node tasks over ``(time, frequency)`` chunks, so the same dataset can be imaged
directly with `image_cube_single_field`.

---
## API


In [ ]:
from astroviper.distributed_applications.simulation import simulate_processing_set

simulate_processing_set?

## Install AstroVIPER

Skip this cell if you don't want to install the latest version of AstroVIPER.

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

## Load packages and start a Dask client

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from astropy.coordinates import SkyCoord

xr.set_options(display_style="html")
ARCSEC_TO_RAD = np.pi / (180 * 3600)

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

## Telescope layout

The CASA/`simobserve` array configuration files ship with AstroVIPER and are read into an MSv4
``antenna_xds``.

In [ ]:
from astroviper.utils.telescope_layout import (
    list_telescope_layouts,
    read_telescope_layout,
)

print([name for name in list_telescope_layouts() if name.startswith("vla")])
antenna_xds = read_telescope_layout("vla.d")
n_antenna = antenna_xds.sizes["antenna_name"]
antenna_xds

## Time, frequency and polarization setup

``n_time_chunks * n_frequency_chunks`` GraphVIPER tasks are created (maximum parallelism).

In [ ]:
time_params = {
    "time_start": "2019-10-03T19:00:00.000",
    "time_delta": 3600.0,
    "n_samples": 10,
}
frequency_params = {
    "freq_start": 3.0e9,
    "freq_delta": 0.4e9,
    "n_channels": 3,
    "channel_width": 0.01e9,
    "spectral_window_name": "SBand",
}
polarization = ["RR", "LL"]

## Beam models

Antenna beams can be analytic Airy disks (parameters taken from CASA's ``PBMath``), beam
polynomials or Zernike aperture coefficients.  ``beam_model_map`` assigns a model to every antenna.

In [ ]:
from astroviper.utils.beam_models import airy_disk_model

airy_vla = airy_disk_model("vla")
print(airy_vla)
beam_models = [airy_vla]
beam_model_map = np.zeros(n_antenna, dtype=int)
beam_params = {}  # defaults: mueller_selection=[0, 5, 10, 15], pa_radius=0.2, ...

## Sources and phase centre

- ``point_source_ra_dec``: ``[n_time | 1, n_source, 2]`` radians
- ``point_source_flux``: ``[n_source, n_time | 1, n_frequency | 1, 4]`` Jy in the instrumental basis (all four values are needed)
- ``phase_center_ra_dec``: ``[n_time | 1, 2]`` radians

In [ ]:
point_source = SkyCoord(ra="19h59m50.51793355s", dec="+40d48m11.3694551s", frame="icrs")
point_source_ra_dec = np.array([point_source.ra.rad, point_source.dec.rad])[
    None, None, :
]
point_source_flux = np.array([1.0, 0, 0, 1.0])[None, None, None, :]

phase_center = SkyCoord(ra="19h59m28.5s", dec="+40d44m01.5s", frame="icrs")
phase_center_ra_dec = np.array([phase_center.ra.rad, phase_center.dec.rad])[None, :]

from astroviper.utils.coordinate_transforms import sin_project

print(
    "source offset (l, m) in arcmin:",
    sin_project(phase_center_ra_dec[0], point_source_ra_dec[0]) / ARCSEC_TO_RAD / 60,
)

## Run the simulation

The simulation writes ``vla_sim.ps.zarr`` and returns timing information.

In [ ]:
PS_STORE = "vla_sim.ps.zarr"

result = simulate_processing_set(
    ps_store=PS_STORE,
    antenna_xds=antenna_xds,
    time_params=time_params,
    frequency_params=frequency_params,
    polarization=polarization,
    point_source_flux=point_source_flux,
    point_source_ra_dec=point_source_ra_dec,
    phase_center_ra_dec=phase_center_ra_dec,
    beam_models=beam_models,
    beam_model_map=beam_model_map,
    beam_params=beam_params,
    noise_params=None,
    n_time_chunks=2,
    n_frequency_chunks=3,
    overwrite=True,
)
result["timing_node_tasks"]

## Validate the processing set against the MSv4 schema

``xradio.schema.check.check_datatree`` checks every dataset of the processing set (coordinates,
dimensions, dtypes and attributes of the main, antenna and field/source datasets) against the
MSv4 schema.  ``simulate_processing_set`` runs this check itself (``check_schema=True``) and logs
a warning on problems; here it is run explicitly so that the result is visible.

In [ ]:
from xradio.measurement_set import open_processing_set
from xradio.schema.check import check_datatree

ps_xdt = open_processing_set(PS_STORE)
issues = check_datatree(ps_xdt)
print(issues)
assert str(issues) == "No schema issues found"
# the same check works on a single MSv4 (the checker dispatches on the ``type`` attribute)
print(check_datatree(ps_xdt[result["ms_name"]]))

## Inspect the processing set

In [ ]:
ps_xdt.xr_ps.summary()

In [ ]:
ms_xds = ps_xdt[result["ms_name"]].ds
ms_xds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
uvw = ms_xds.UVW.values
axes[0].scatter(uvw[..., 0].ravel(), uvw[..., 1].ravel(), s=1)
axes[0].scatter(-uvw[..., 0].ravel(), -uvw[..., 1].ravel(), s=1)
axes[0].set_xlabel("u [m]")
axes[0].set_ylabel("v [m]")
axes[0].set_title("uv coverage")
axes[0].set_aspect("equal")
amplitude = np.abs(ms_xds.VISIBILITY.isel(frequency=0, polarization=0).values)
axes[1].plot((ms_xds.time.values - ms_xds.time.values[0]) / 3600, amplitude, ".", ms=2)
axes[1].set_xlabel("time [h]")
axes[1].set_ylabel("|V| [Jy]")
axes[1].set_title("amplitude vs time (all baselines, channel 0, RR)")
plt.show()

## Image the simulated dataset

The point source is attenuated by the primary beam (the source sits ~6 arcmin from the phase
centre); the dirty image peak equals the beam-attenuated flux.

In [ ]:
from xradio.image import load_image
from xradio.measurement_set import open_processing_set

from astroviper.distributed_applications.imaging import image_cube_single_field


def image_simulation(
    ps_store,
    image_store,
    image_size,
    cell_size_arcsec,
    niter=0,
    polarization_coords=("I",),
    n_chunks=2,
):
    """Make a (dirty or cleaned) cube of a simulated processing set with AstroVIPER."""
    ps_xdt = open_processing_set(ps_store)
    combined = ps_xdt.xr_ps.get_combined_field_and_source_xds()
    phase_direction = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=combined.attrs["center_field_name"]
    ).values
    image_params = {
        "image_size": list(image_size),
        "cell_size": np.array([-cell_size_arcsec, cell_size_arcsec]) * ARCSEC_TO_RAD,
        "phase_direction": phase_direction,
        "frequency_coords": ps_xdt.xr_ps.get_freq_axis().values,
        "polarization_coords": list(polarization_coords),
        "time_coords": [0],
        "fft_padding": 1.2,
        "cpp_gridder": True,
    }
    iteration_control = {
        "niter": niter,
        "nmajor": -1 if niter > 0 else 0,
        "threshold": 0.0,
        "gain": 0.1,
        "cyclefactor": 1.5,
        "cycleniter": -1,
        "minpsffraction": 0.05,
        "maxpsffraction": 0.8,
        "primary_beam_limit": 0.1,
    }
    keep = [
        "sky_residual",
        "point_spread_function",
        "primary_beam",
        "beam_fit_params_point_spread_function",
    ]
    if niter > 0:
        keep += ["sky_model", "mask"]
    image_cube_single_field(
        ps_store=ps_store,
        image_store=image_store,
        image_params=image_params,
        imaging_weights_params={
            "weighting": "natural",
            "robust": 0.5,
            "casa_weighting_implementation": True,
        },
        iteration_control_params=iteration_control,
        gridder="prolate_spheroidal",
        deconvolver="hogbom_many_threads",
        scan_intents="OBSERVE_TARGET#ON_SOURCE",
        image_data_variables_keep=keep,
        processing_set_data_group_name="base",
        single_precision_image=False,
        processing_function_threads=1,
        n_chunks=n_chunks,
        overwrite=True,
        restore=niter > 0,
    )
    return load_image(image_store)


def show_image(
    img_xds, variable="SKY_RESIDUAL", frequency=0, polarization=0, title=None, vmax=None
):
    """Plot one plane of an AstroVIPER image with l/m in arcsec."""
    plane = (
        img_xds[variable]
        .isel(time=0, frequency=frequency, polarization=polarization)
        .values
    )
    extent = (
        np.array(
            [
                img_xds.l.values[0],
                img_xds.l.values[-1],
                img_xds.m.values[0],
                img_xds.m.values[-1],
            ]
        )
        / ARCSEC_TO_RAD
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(plane.T, origin="lower", extent=extent, cmap="viridis", vmax=vmax)
    ax.set_xlabel("l [arcsec]")
    ax.set_ylabel("m [arcsec]")
    ax.set_title(
        title
        or f"{variable} channel {frequency} ({img_xds.frequency.values[frequency] / 1e9:.3f} GHz)"
    )
    fig.colorbar(im, ax=ax, label="Jy/beam")
    return fig

In [ ]:
img_xds = image_simulation(
    PS_STORE, "vla_sim.img.zarr", image_size=[400, 400], cell_size_arcsec=5.0, niter=0
)
show_image(img_xds, frequency=0)
plt.show()

from astroviper.utils.coordinate_transforms import celestial_coord_to_sin_pixel

expected_pixel = celestial_coord_to_sin_pixel(
    phase_center_ra_dec[0],
    [400, 400],
    np.array([-5.0, 5.0]) * ARCSEC_TO_RAD,
    point_source_ra_dec[0, 0],
)
plane = img_xds.SKY_RESIDUAL.isel(time=0, frequency=0, polarization=0).values
print(
    "expected source pixel",
    expected_pixel,
    "image peak pixel",
    np.unravel_index(np.argmax(plane), plane.shape),
    "peak",
    plane.max(),
)

## Variant: EVLA beam polynomial model

The SIRIUS `evla_simulation` example uses CASA's EVLA ``PBMath1DPoly`` beam polynomials instead of
an Airy disk.

In [ ]:
from astroviper.utils.beam_models import read_beam_polynomial_coefficients

bpc_xds = read_beam_polynomial_coefficients("EVLA_")
bpc_xds

In [ ]:
result_poly = simulate_processing_set(
    ps_store="vla_sim_poly.ps.zarr",
    antenna_xds=antenna_xds,
    time_params=time_params,
    frequency_params=frequency_params,
    polarization=polarization,
    point_source_flux=point_source_flux,
    point_source_ra_dec=point_source_ra_dec,
    phase_center_ra_dec=phase_center_ra_dec,
    beam_models=[bpc_xds],
    beam_model_map=beam_model_map,
    n_time_chunks=2,
    n_frequency_chunks=3,
    overwrite=True,
)
print(check_datatree(open_processing_set("vla_sim_poly.ps.zarr")))
ms_poly = open_processing_set("vla_sim_poly.ps.zarr")[result_poly["ms_name"]].ds
print("mean |V| airy      :", np.abs(ms_xds.VISIBILITY.values).mean(axis=(0, 1, 3)))
print("mean |V| polynomial:", np.abs(ms_poly.VISIBILITY.values).mean(axis=(0, 1, 3)))

## Clean up

In [ ]:
import shutil

for path in [PS_STORE, "vla_sim_poly.ps.zarr", "vla_sim.img.zarr"]:
    shutil.rmtree(path, ignore_errors=True)
viper_client.close()